In [1]:
# ==========================================
# STEP 1: IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
import torch

from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.1+cu132
GPU available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
# ==========================================
# STEP 2: LOAD EMOTION DATASETS
# ==========================================

DATA_DIR = Path("../data/processed/emotion")

train_df = pd.read_csv(DATA_DIR / "emotion_train.csv")
dev_df = pd.read_csv(DATA_DIR / "emotion_dev.csv")
test_df = pd.read_csv(DATA_DIR / "emotion_test.csv")

print("Train:", train_df.shape)
print("Validation:", dev_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (9989, 13)
Validation: (1109, 13)
Test: (2610, 13)


,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime,clean_text,emotion_id
0,1,also I was the point person on my company’s tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731",also i was the point person on my company’s tr...,4
1,2,You must’ve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442",you must’ve had your hands full.,4
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389",that i did. that i did.,4
3,4,So let’s talk a little bit about your duties.,The Interviewer,neutral,neutral,0,3,8,21,"00:16:26,820","00:16:29,572",so let’s talk a little bit about your duties.,4
4,5,My duties? All right.,Chandler,surprise,positive,0,4,8,21,"00:16:34,452","00:16:40,917",my duties? all right.,6


In [3]:
# ==========================================
# STEP 3: CHECK EMOTION LABELS
# ==========================================

EMOTION_LABELS = [
    "anger", "disgust", "fear", "joy",
    "neutral", "sadness", "surprise"
]

print("Train distribution:")
print(train_df["Emotion"].value_counts())

print("\nValidation distribution:")
print(dev_df["Emotion"].value_counts())

print("\nTest distribution:")
print(test_df["Emotion"].value_counts())

Train distribution:
Emotion
neutral     4710
joy         1743
surprise    1205
anger       1109
sadness      683
disgust      271
fear         268
Name: count, dtype: int64

Validation distribution:
Emotion
neutral     470
joy         163
anger       153
surprise    150
sadness     111
fear         40
disgust      22
Name: count, dtype: int64

Test distribution:
Emotion
neutral     1256
joy          402
anger        345
surprise     281
sadness      208
disgust       68
fear          50
Name: count, dtype: int64


In [4]:
# ==========================================
# STEP 4: TRANSFORMER LIBRARIES
# ==========================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

print("Transformer libraries ready!")

Transformer libraries ready!


In [5]:
# ==========================================
# STEP 5: MODEL CONFIGURATION
# ==========================================

MODEL_NAME = "distilbert-base-uncased"

NUM_LABELS = 7

id2label = {
    0: "anger",
    1: "disgust",
    2: "fear",
    3: "joy",
    4: "neutral",
    5: "sadness",
    6: "surprise"
}

label2id = {
    label: idx for idx, label in id2label.items()
}

MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Model:", MODEL_NAME)

Device: cuda
Model: distilbert-base-uncased


In [6]:
# ==========================================
# STEP 6: TOKENIZATION
# ==========================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_emotions(batch):
    return tokenizer(
        batch["clean_text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_dataset = Dataset.from_pandas(
    train_df[["clean_text", "emotion_id"]]
).rename_column("emotion_id", "labels")

dev_dataset = Dataset.from_pandas(
    dev_df[["clean_text", "emotion_id"]]
).rename_column("emotion_id", "labels")

test_dataset = Dataset.from_pandas(
    test_df[["clean_text", "emotion_id"]]
).rename_column("emotion_id", "labels")

train_dataset = train_dataset.map(tokenize_emotions, batched=True)
dev_dataset = dev_dataset.map(tokenize_emotions, batched=True)
test_dataset = test_dataset.map(tokenize_emotions, batched=True)

columns_to_keep = ["input_ids", "attention_mask", "labels"]

train_dataset.set_format(type="torch", columns=columns_to_keep)
dev_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

print("Tokenization complete!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\sansm\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sansm\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/9989 [00:00<?, ? examples/s]

Map:   0%|          | 0/1109 [00:00<?, ? examples/s]

Map:   0%|          | 0/2610 [00:00<?, ? examples/s]

Tokenization complete!


In [7]:
# ==========================================
# STEP 7: LOAD EMOTION MODEL
# ==========================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

model.to(DEVICE)

print("Emotion model loaded!")

c:\Users\sansm\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sansm\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Emotion model loaded!


In [8]:
# ==========================================
# STEP 8: EVALUATION METRICS
# ==========================================

from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted")
    }

In [9]:
# ==========================================
# STEP 9: TRAINING CONFIGURATION
# ==========================================

OUTPUT_DIR = "../models/distilbert_meld_emotion"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
    seed=42
)

print("Training configuration ready!")

Training configuration ready!


In [10]:
# ==========================================
# STEP 10: TRAIN EMOTION CLASSIFIER
# ==========================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.116535,1.205538,0.601443,0.367623,0.563115
2,1.020075,1.163115,0.606853,0.383331,0.581085
3,0.895277,1.191912,0.608656,0.382260,0.577301


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=1.0516313720703125, metrics={'train_runtime': 109.2393, 'train_samples_per_second': 274.324, 'train_steps_per_second': 17.164, 'total_flos': 992501125233408.0, 'train_loss': 1.0516313720703125, 'epoch': 3.0})

In [11]:
# ==========================================
# STEP 11: TEST SET EVALUATION
# ==========================================

test_results = trainer.evaluate(test_dataset)

print("\nFINAL EMOTION MODEL RESULTS")
print("=" * 50)

for key, value in test_results.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.4f}")

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1
0.895277,1.068991,3,0.639464,0.392547,0.620390



FINAL EMOTION MODEL RESULTS
eval_loss: 1.0690
eval_accuracy: 0.6395
eval_macro_f1: 0.3925
eval_weighted_f1: 0.6204


In [12]:
# ==========================================
# STEP 12: CLASSIFICATION REPORT
# ==========================================

pred_output = trainer.predict(test_dataset)

y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print(classification_report(
    y_true,
    y_pred,
    target_names=EMOTION_LABELS,
    digits=4
))

              precision    recall  f1-score   support

       anger     0.4529    0.5014    0.4759       345
     disgust     0.0000    0.0000    0.0000        68
        fear     0.0000    0.0000    0.0000        50
         joy     0.5693    0.5721    0.5707       402
     neutral     0.7571    0.8312    0.7924      1256
     sadness     0.3801    0.3125    0.3430       208
    surprise     0.5730    0.5587    0.5658       281

    accuracy                         0.6395      2610
   macro avg     0.3903    0.3966    0.3925      2610
weighted avg     0.6039    0.6395    0.6204      2610



c:\Users\sansm\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sansm\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\sansm\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [13]:
# ==========================================
# STEP 13: SAVE FINAL MODEL
# ==========================================

SAVE_PATH = Path("../models/distilbert_meld_emotion")
SAVE_PATH.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(SAVE_PATH))
tokenizer.save_pretrained(str(SAVE_PATH))

print("Emotion model saved at:", SAVE_PATH.resolve())

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Emotion model saved at: C:\Users\sansm\Desktop\NLP PROJECT CHATGPT\Conversation_Sentiment_Analysis\models\distilbert_meld_emotion
